### **download kaggle dataset into local  colab filesystem**

In [117]:
mkdir -p /content/kaggledata

In [118]:
mkdir /content/kaggleresources


mkdir: cannot create directory ‘/content/kaggleresources’: File exists


In [119]:
import os
from google.colab import userdata

# Configure Kaggle API credentials using Colab secrets
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("Kaggle API credentials configured successfully.")
except Exception as e:
    print(f"Error fetching secrets: {e}")
    print("Please ensure you have set KAGGLE_USERNAME and KAGGLE_KEY in the Colab Secrets panel (the key icon on the left).")

Kaggle API credentials configured successfully.


In [120]:
#!/bin/bash
!kaggle datasets download  joycemara/european-fashion-store-multitable-dataset -p  /content/kaggledata

Dataset URL: https://www.kaggle.com/datasets/joycemara/european-fashion-store-multitable-dataset
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
european-fashion-store-multitable-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [121]:
!unzip -o /content/kaggledata/european-fashion-store-multitable-dataset.zip -d /content/kaggleresources

Archive:  /content/kaggledata/european-fashion-store-multitable-dataset.zip
  inflating: /content/kaggleresources/dataset_fashion_store_campaigns.csv  
  inflating: /content/kaggleresources/dataset_fashion_store_channels.csv  
  inflating: /content/kaggleresources/dataset_fashion_store_customers.csv  
  inflating: /content/kaggleresources/dataset_fashion_store_products.csv  
  inflating: /content/kaggleresources/dataset_fashion_store_sales.csv  
  inflating: /content/kaggleresources/dataset_fashion_store_salesitems.csv  
  inflating: /content/kaggleresources/dataset_fashion_store_stock.csv  


### **data ingestion and exploration**

In [122]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("kaggle-european-fashion-store-multitable-dataset").getOrCreate()

In [123]:
fashion_store_campaigns_csv_path="/content/kaggleresources/dataset_fashion_store_campaigns.csv"
fashion_store_channels_csv_path="/content/kaggleresources/dataset_fashion_store_channels.csv"
fashion_store_customers_csv_path="/content/kaggleresources/dataset_fashion_store_customers.csv"
fashion_store_products_csv_path="/content/kaggleresources/dataset_fashion_store_products.csv"
fashion_store_sales_csv_path="/content/kaggleresources/dataset_fashion_store_sales.csv"
fashion_store_salesitems_csv_path="/content/kaggleresources/dataset_fashion_store_salesitems.csv"
fashion_store_stock_csv_path="/content/kaggleresources/dataset_fashion_store_stock.csv"

In [124]:
fashion_store_campaigns_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_campaigns_csv_path)
fashion_store_channels_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_channels_csv_path)
fashion_store_customers_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_customers_csv_path)
fashion_store_products_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_products_csv_path)
fashion_store_sales_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_sales_csv_path)
fashion_store_salesitems_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_salesitems_csv_path)
fashion_store_stock_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_stock_csv_path)

In [125]:
fashion_store_campaigns_df.show()
fashion_store_channels_df.show()
fashion_store_customers_df.show()
fashion_store_products_df.show()
fashion_store_sales_df.show()
fashion_store_salesitems_df.show()
fashion_store_stock_df.show()

+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|campaign_id|       campaign_name|start_date|  end_date|       channel|discount_type|discount_value|
+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|          1|   Spring Flash Sale|2025-04-01|2025-04-07|         Email|   Percentage|        10.00%|
|          2|    Easter Promotion|2025-04-08|2025-04-15|  Social Media|        Fixed|            10|
|          3|Mother's Day Camp...|2025-05-01|2025-05-09|         Email|   Percentage|        25.00%|
|          4|Mid-Season Clearance|2025-05-10|2025-05-19|    App Mobile|   Percentage|        30.00%|
|          5|           TIVA Week|2025-05-20|2025-05-31|  Social Media|   Percentage|        30.00%|
|          6|     June Price Drop|2025-06-01|2025-06-09|Website Banner|   Percentage|        10.00%|
|          7|  Early Summer Deals|2025-06-10|2025-06-17|         Email|        Fixed|      

In [126]:
fashion_store_campaigns_df.printSchema()
fashion_store_channels_df.printSchema()
fashion_store_customers_df.printSchema()
fashion_store_products_df.printSchema()
fashion_store_sales_df.printSchema()
fashion_store_salesitems_df.printSchema()
fashion_store_stock_df.printSchema()

root
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- channel: string (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)

root
 |-- channel: string (nullable = true)
 |-- description: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- age_range: string (nullable = true)
 |-- signup_date: date (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- color: string (nullable = true)
 |-- size: string (nullable = true)
 |-- catalog_price: double (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- gender: string (nullable = true)

root
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nul

### **handling null values**

In [127]:
from pyspark.sql.functions import isnan,isnull, when, count, col

In [128]:
fashion_store_campaigns_df.select([count(when(isnull(c),c)).alias(c) for c in fashion_store_campaigns_df.columns]).show()
fashion_store_channels_df.select([count(when(isnull(c),c)).alias(c) for c in fashion_store_channels_df.columns]).show()
fashion_store_customers_df.select([count(when(isnull(c),c)).alias(c) for c in fashion_store_customers_df.columns]).show()
fashion_store_products_df.select([count(when(isnull(c),c)).alias(c) for c in fashion_store_products_df.columns]).show()
fashion_store_sales_df.select([count(when(isnull(c),c)).alias(c) for c in fashion_store_sales_df.columns]).show()
fashion_store_salesitems_df.select([count(when(isnull(c),c)).alias(c) for c in fashion_store_salesitems_df.columns]).show()
fashion_store_stock_df.select([count(when(isnull(c),c)).alias(c) for c in fashion_store_stock_df.columns]).show()

+-----------+-------------+----------+--------+-------+-------------+--------------+
|campaign_id|campaign_name|start_date|end_date|channel|discount_type|discount_value|
+-----------+-------------+----------+--------+-------+-------------+--------------+
|          0|            0|         0|       0|      0|            0|             0|
+-----------+-------------+----------+--------+-------+-------------+--------------+

+-------+-----------+
|channel|description|
+-------+-----------+
|      0|          0|
+-------+-----------+

+-----------+-------+---------+-----------+
|customer_id|country|age_range|signup_date|
+-----------+-------+---------+-----------+
|          0|      0|        0|          0|
+-----------+-------+---------+-----------+

+----------+------------+--------+-----+-----+----+-------------+----------+------+
|product_id|product_name|category|brand|color|size|catalog_price|cost_price|gender|
+----------+------------+--------+-----+-----+----+-------------+---------

### **checking dupplicate values accross columns**

In [129]:
fashion_store_campaigns_df.groupBy("campaign_id").count().filter("count>1").show()
fashion_store_campaigns_df.groupBy("discount_type").count().filter("count>1").show()
fashion_store_campaigns_df.groupBy("channel").count().filter("count>1").show()


+-----------+-----+
|campaign_id|count|
+-----------+-----+
+-----------+-----+

+-------------+-----+
|discount_type|count|
+-------------+-----+
|        Fixed|    2|
|   Percentage|    5|
+-------------+-----+

+------------+-----+
|     channel|count|
+------------+-----+
|       Email|    3|
|Social Media|    2|
+------------+-----+



In [130]:
fashion_store_channels_df.groupBy("channel").count().filter("count>1").show()
fashion_store_channels_df.groupBy("description").count().filter("count>1").show()


+-------+-----+
|channel|count|
+-------+-----+
+-------+-----+

+-----------+-----+
|description|count|
+-----------+-----+
+-----------+-----+



In [131]:
fashion_store_customers_df.groupBy("customer_id").count().filter("count>1").show()
fashion_store_customers_df.groupBy("country").count().filter("count>1").show()
fashion_store_customers_df.groupBy("age_range").count().filter("count>1").show()
fashion_store_customers_df.groupBy("signup_date").count().filter("count>1").show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+

+-----------+-----+
|    country|count|
+-----------+-----+
|    Germany|  212|
|     France|  221|
|      Italy|  192|
|      Spain|  143|
|   Portugal|   70|
|Netherlands|  162|
+-----------+-----+

+---------+-----+
|age_range|count|
+---------+-----+
|    26-35|  206|
|    16-25|  207|
|    46-55|  198|
|    36-45|  204|
|    56-65|  185|
+---------+-----+

+-----------+-----+
|signup_date|count|
+-----------+-----+
| 2025-02-16|    4|
| 2025-02-15|    6|
| 2024-11-02|    2|
| 2025-02-01|    4|
| 2025-03-23|    4|
| 2025-04-17|    6|
| 2025-02-05|    5|
| 2025-02-13|    6|
| 2024-07-08|    3|
| 2025-05-13|    3|
| 2024-09-10|    2|
| 2025-03-16|    6|
| 2024-02-04|    3|
| 2024-08-05|    3|
| 2025-02-06|    6|
| 2024-08-12|    3|
| 2025-02-12|    4|
| 2024-08-14|    2|
| 2024-07-05|    3|
| 2025-04-21|    7|
+-----------+-----+
only showing top 20 rows


In [132]:
fashion_store_products_df.groupBy("product_id").count().filter("count>1").show()
fashion_store_products_df.groupBy("product_name").count().filter("count>1").show()
fashion_store_products_df.groupBy("category").count().filter("count>1").show()
fashion_store_products_df.groupBy("brand").count().filter("count>1").show()
fashion_store_products_df.groupBy("color").count().filter("count>1").show()
fashion_store_products_df.groupBy("size").count().filter("count>1").show()
fashion_store_products_df.groupBy("catalog_price").count().filter("count>1").show()
fashion_store_products_df.groupBy("cost_price").count().filter("count>1").show()
fashion_store_products_df.groupBy("gender").count().filter("count>1").show()

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+

+------------+-----+
|product_name|count|
+------------+-----+
+------------+-----+

+---------+-----+
| category|count|
+---------+-----+
|  Dresses|  109|
| T-Shirts|  108|
|Sleepwear|  104|
|    Pants|   79|
|    Shoes|  100|
+---------+-----+

+-----+-----+
|brand|count|
+-----+-----+
| Tiva|  500|
+-----+-----+

+-----+-----+
|color|count|
+-----+-----+
|Green|  102|
| Blue|  102|
|White|   94|
|Black|  104|
|  Red|   98|
+-----+-----+

+----+-----+
|size|count|
+----+-----+
|  XL|   81|
|  35|   16|
|   M|   72|
|   L|   76|
|  40|   19|
|  38|   20|
|   S|   84|
|  XS|  107|
|  36|   25|
+----+-----+

+-------------+-----+
|catalog_price|count|
+-------------+-----+
|        39.18|    2|
|         46.9|    2|
|        58.11|    2|
|        50.34|    2|
|        52.28|    2|
|        28.98|    2|
|        64.08|    2|
|        24.05|    2|
|        48.73|    2|
|         59.9|    2|
|        60.33|    2|

In [133]:
fashion_store_sales_df.groupBy("sale_id").count().filter("count>1").show()
fashion_store_sales_df.groupBy("channel").count().filter("count>1").show()
fashion_store_sales_df.groupBy("discounted").count().filter("count>1").show()
fashion_store_sales_df.groupBy("total_amount").count().filter("count>1").show()
fashion_store_sales_df.groupBy("sale_date").count().filter("count>1").show()
fashion_store_sales_df.groupBy("customer_id").count().filter("count>1").show()
fashion_store_sales_df.groupBy("country").count().filter("count>1").show()

+-------+-----+
|sale_id|count|
+-------+-----+
+-------+-----+

+----------+-----+
|   channel|count|
+----------+-----+
|App Mobile|  432|
|E-commerce|  473|
+----------+-----+

+----------+-----+
|discounted|count|
+----------+-----+
|         1|   89|
|         0|  816|
+----------+-----+

+------------+-----+
|total_amount|count|
+------------+-----+
|      232.25|    2|
|      223.28|    2|
|      535.45|    2|
|       282.2|    2|
|      181.08|    2|
|      390.32|    2|
|      309.83|    2|
+------------+-----+

+----------+-----+
| sale_date|count|
+----------+-----+
|2025-04-17|   19|
|2025-05-13|   20|
|2025-05-15|   18|
|2025-05-30|   13|
|2025-04-21|   20|
|2025-06-05|   15|
|2025-05-26|   18|
|2025-06-16|   22|
|2025-05-27|   19|
|2025-04-23|   15|
|2025-05-24|   18|
|2025-05-20|   21|
|2025-04-16|   16|
|2025-05-31|   25|
|2025-04-05|   18|
|2025-04-13|   12|
|2025-05-29|   14|
|2025-04-20|   17|
|2025-05-23|   18|
|2025-05-21|   22|
+----------+-----+
only showing top 

In [134]:
fashion_store_salesitems_df.groupBy("item_id").count().filter("count>1").show()
fashion_store_salesitems_df.groupBy("sale_id").count().filter("count>1").show()
fashion_store_salesitems_df.groupBy("product_id").count().filter("count>1").show()
fashion_store_salesitems_df.groupBy("quantity").count().filter("count>1").show()
fashion_store_salesitems_df.groupBy("original_price").count().filter("count>1").show()
fashion_store_salesitems_df.groupBy("unit_price").count().filter("count>1").show()
fashion_store_salesitems_df.groupBy("discount_applied").count().filter("count>1").show()
fashion_store_salesitems_df.groupBy("discount_percent").count().filter("count>1").show()
fashion_store_salesitems_df.groupBy("discounted").count().filter("count>1").show()
fashion_store_salesitems_df.groupBy("item_total").count().filter("count>1").show()
fashion_store_salesitems_df.groupBy("sale_date").count().filter("count>1").show()
fashion_store_salesitems_df.groupBy("channel").count().filter("count>1").show()
fashion_store_salesitems_df.groupBy("channel_campaigns").count().filter("count>1").show()

+-------+-----+
|item_id|count|
+-------+-----+
+-------+-----+

+-------+-----+
|sale_id|count|
+-------+-----+
|    833|    2|
|   1088|    2|
|    463|    3|
|   1342|    2|
|   1025|    2|
|    623|    3|
|    392|    3|
|    858|    2|
|   1127|    2|
|   1084|    2|
|    737|    2|
|     31|    3|
|   1339|    2|
|   1143|    2|
|   1322|    2|
|    516|    3|
|   1352|    2|
|   1139|    2|
|   1270|    2|
|   1303|    2|
+-------+-----+
only showing top 20 rows
+----------+-----+
|product_id|count|
+----------+-----+
|       463|    4|
|       471|    4|
|       496|    6|
|       148|    6|
|       392|    5|
|       243|    5|
|        31|    4|
|       137|    8|
|       251|    6|
|       451|    5|
|        85|    5|
|       458|    7|
|        65|    4|
|        53|    7|
|       481|    6|
|       255|    6|
|       472|    5|
|       133|    6|
|       296|    6|
|       322|    7|
+----------+-----+
only showing top 20 rows
+--------+-----+
|quantity|count|
+--------+-

In [135]:
fashion_store_stock_df.groupBy("country").count().filter("count>1").show()
fashion_store_stock_df.groupBy("product_id").count().filter("count>1").show()
fashion_store_stock_df.groupBy("stock_quantity").count().filter("count>1").show()


+-------+-----+
|country|count|
+-------+-----+
|Germany|  500|
| France|  500|
+-------+-----+

+----------+-----+
|product_id|count|
+----------+-----+
|       148|    2|
|       463|    2|
|       471|    2|
|       496|    2|
|       243|    2|
|       392|    2|
|        31|    2|
|        85|    2|
|       137|    2|
|       251|    2|
|       451|    2|
|        65|    2|
|       458|    2|
|        53|    2|
|       255|    2|
|       481|    2|
|       133|    2|
|       296|    2|
|       472|    2|
|        78|    2|
+----------+-----+
only showing top 20 rows
+--------------+-----+
|stock_quantity|count|
+--------------+-----+
|            31|    5|
|            85|    5|
|            65|    2|
|            53|    8|
|            78|    8|
|            34|    3|
|            81|    4|
|            28|    8|
|            76|    6|
|            27|    4|
|            26|    5|
|            44|    6|
|            12|    6|
|            91|    3|
|            22|    3|
|       

### **Aggrigations and Grouping**

**grouping the camplaings based on channel**

In [136]:
fashion_store_campaigns_df.groupBy("channel").count().orderBy("count",ascending=False).show()
fashion_store_campaigns_df.groupBy("channel").count().orderBy("count").show()


+--------------+-----+
|       channel|count|
+--------------+-----+
|         Email|    3|
|  Social Media|    2|
|    App Mobile|    1|
|Website Banner|    1|
+--------------+-----+

+--------------+-----+
|       channel|count|
+--------------+-----+
|    App Mobile|    1|
|Website Banner|    1|
|  Social Media|    2|
|         Email|    3|
+--------------+-----+



In [137]:
fashion_store_customers_df.groupBy("country").count().orderBy(col("count").desc()).show()
fashion_store_customers_df.groupBy("country").count().orderBy("count").show()

+-----------+-----+
|    country|count|
+-----------+-----+
|     France|  221|
|    Germany|  212|
|      Italy|  192|
|Netherlands|  162|
|      Spain|  143|
|   Portugal|   70|
+-----------+-----+

+-----------+-----+
|    country|count|
+-----------+-----+
|   Portugal|   70|
|      Spain|  143|
|Netherlands|  162|
|      Italy|  192|
|    Germany|  212|
|     France|  221|
+-----------+-----+



In [138]:
fashion_store_products_df.groupBy("color").count().orderBy(col("count").desc()).show()
fashion_store_products_df.groupBy("color").count().orderBy("count").show()
fashion_store_products_df.groupBy("size").count().orderBy(col("count").desc()).show()
fashion_store_products_df.groupBy("size").count().orderBy("count").show()

+-----+-----+
|color|count|
+-----+-----+
|Black|  104|
|Green|  102|
| Blue|  102|
|  Red|   98|
|White|   94|
+-----+-----+

+-----+-----+
|color|count|
+-----+-----+
|White|   94|
|  Red|   98|
|Green|  102|
| Blue|  102|
|Black|  104|
+-----+-----+

+----+-----+
|size|count|
+----+-----+
|  XS|  107|
|   S|   84|
|  XL|   81|
|   L|   76|
|   M|   72|
|  36|   25|
|  38|   20|
|  40|   19|
|  35|   16|
+----+-----+

+----+-----+
|size|count|
+----+-----+
|  35|   16|
|  40|   19|
|  38|   20|
|  36|   25|
|   M|   72|
|   L|   76|
|  XL|   81|
|   S|   84|
|  XS|  107|
+----+-----+



In [139]:
fashion_store_sales_df.groupBy("country").count().orderBy(col("count").desc()).show()
fashion_store_sales_df.groupBy("country").count().orderBy("count").show()

+-----------+-----+
|    country|count|
+-----------+-----+
|    Germany|  212|
|     France|  199|
|      Italy|  170|
|Netherlands|  131|
|      Spain|  113|
|   Portugal|   80|
+-----------+-----+

+-----------+-----+
|    country|count|
+-----------+-----+
|   Portugal|   80|
|      Spain|  113|
|Netherlands|  131|
|      Italy|  170|
|     France|  199|
|    Germany|  212|
+-----------+-----+



**items sold based on sale date**

In [140]:
fashion_store_salesitems_df.groupBy("sale_date").count().orderBy(col("count").desc()).show()
fashion_store_salesitems_df.groupBy("sale_date").count().orderBy("count").show()

+----------+-----+
| sale_date|count|
+----------+-----+
|2025-05-31|   63|
|2025-05-18|   62|
|2025-06-14|   61|
|2025-05-20|   57|
|2025-04-19|   55|
|2025-05-21|   55|
|2025-05-13|   53|
|2025-04-18|   53|
|2025-06-16|   52|
|2025-05-19|   52|
|2025-04-22|   51|
|2025-05-17|   50|
|2025-04-28|   49|
|2025-04-21|   49|
|2025-05-15|   49|
|2025-05-26|   48|
|2025-05-04|   47|
|2025-05-16|   47|
|2025-05-22|   46|
|2025-04-25|   46|
+----------+-----+
only showing top 20 rows
+----------+-----+
| sale_date|count|
+----------+-----+
|2025-06-05|   28|
|2025-04-13|   29|
|2025-04-06|   31|
|2025-05-30|   33|
|2025-06-15|   34|
|2025-04-23|   36|
|2025-04-16|   36|
|2025-05-29|   36|
|2025-04-04|   36|
|2025-04-29|   37|
|2025-06-04|   37|
|2025-04-24|   37|
|2025-05-28|   37|
|2025-05-05|   38|
|2025-05-06|   38|
|2025-04-05|   39|
|2025-05-25|   39|
|2025-06-13|   40|
|2025-04-15|   41|
|2025-06-17|   42|
+----------+-----+
only showing top 20 rows


**products exist in stock based on country**

In [141]:
fashion_store_stock_df.groupBy("country").count().orderBy(col("count").desc()).show()
fashion_store_stock_df.groupBy("country").count().orderBy("count").show()

+-------+-----+
|country|count|
+-------+-----+
|Germany|  500|
| France|  500|
+-------+-----+

+-------+-----+
|country|count|
+-------+-----+
|Germany|  500|
| France|  500|
+-------+-----+



**finding top selling products**

In [142]:
from pyspark.sql.functions import sum

# Aggregate total quantity sold by product_id
top_selling_df = fashion_store_salesitems_df.groupBy("product_id") \
    .agg(sum("quantity").alias("total_quantity_sold"))

top_selling_df.show()

# Join with the products DataFrame to get the product details
top_selling_products = top_selling_df.join(fashion_store_products_df, "product_id") \
    .select("product_id", "product_name", "category", "brand", "total_quantity_sold") \
    .orderBy(col("total_quantity_sold").desc())

# Show the top selling products
top_selling_products.show(10, truncate=False)

+----------+-------------------+
|product_id|total_quantity_sold|
+----------+-------------------+
|       463|                 13|
|       471|                  5|
|       496|                 20|
|       148|                 15|
|       392|                 15|
|       243|                 12|
|        31|                  7|
|       137|                 23|
|       251|                 18|
|       451|                 15|
|        85|                 17|
|       458|                 15|
|        65|                 16|
|        53|                 25|
|       481|                 15|
|       255|                 13|
|       472|                 13|
|       133|                 18|
|       296|                 20|
|       322|                 22|
+----------+-------------------+
only showing top 20 rows
+----------+-----------------------+---------+-----+-------------------+
|product_id|product_name           |category |brand|total_quantity_sold|
+----------+-----------------------+-

In [143]:
top_selling_products.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_quantity_sold#23435L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_quantity_sold#23435L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=35351]
      +- Project [product_id#8352, product_name#8301, category#8302, brand#8303, total_quantity_sold#23435L]
         +- BroadcastHashJoin [product_id#8352], [product_id#8300], Inner, BuildRight, false
            :- HashAggregate(keys=[product_id#8352], functions=[sum(quantity#8353)])
            :  +- Exchange hashpartitioning(product_id#8352, 200), ENSURE_REQUIREMENTS, [plan_id=35344]
            :     +- HashAggregate(keys=[product_id#8352], functions=[partial_sum(quantity#8353)])
            :        +- Filter isnotnull(product_id#8352)
            :           +- InMemoryTableScan [product_id#8352, quantity#8353], [isnotnull(product_id#8352)]
            :                 +- InMemoryRelation [item_id#8350, sale_id#8351, product_id#8352, quantity#8

In [144]:
top_selling_products.explain(extended=True)

== Parsed Logical Plan ==
'Sort ['total_quantity_sold DESC NULLS LAST], true
+- Project [product_id#8352, product_name#8301, category#8302, brand#8303, total_quantity_sold#23435L]
   +- Project [product_id#8352, total_quantity_sold#23435L, product_name#8301, category#8302, brand#8303, color#8304, size#8305, catalog_price#8306, cost_price#8307, gender#8308]
      +- Join Inner, (product_id#8352 = product_id#8300)
         :- Aggregate [product_id#8352], [product_id#8352, sum(quantity#8353) AS total_quantity_sold#23435L]
         :  +- Relation [item_id#8350,sale_id#8351,product_id#8352,quantity#8353,original_price#8354,unit_price#8355,discount_applied#8356,discount_percent#8357,discounted#8358,item_total#8359,sale_date#8360,channel#8361,channel_campaigns#8362] csv
         +- Relation [product_id#8300,product_name#8301,category#8302,brand#8303,color#8304,size#8305,catalog_price#8306,cost_price#8307,gender#8308] csv

== Analyzed Logical Plan ==
product_id: int, product_name: string, cate

In [145]:
top_selling_products.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- total_quantity_sold: long (nullable = true)



In [146]:
top_selling_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- total_quantity_sold: long (nullable = true)



In [147]:
joined_df=fashion_store_salesitems_df.join(fashion_store_products_df, "product_id")

In [148]:
joined_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- channel: string (nullable = true)
 |-- channel_campaigns: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- color: string (nullable = true)
 |-- size: string (nullable = true)
 |-- catalog_price: double (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- gender: string (nullable = true)



In [149]:
joined_df=joined_df.withColumn("total_amount", col("original_price") + col("unit_price"))

In [150]:
joined_df.show()


+----------+-------+-------+--------+--------------+----------+----------------+----------------+----------+----------+----------+----------+-----------------+--------------------+---------+-----+-----+----+-------------+----------+------+------------+
|product_id|item_id|sale_id|quantity|original_price|unit_price|discount_applied|discount_percent|discounted|item_total| sale_date|   channel|channel_campaigns|        product_name| category|brand|color|size|catalog_price|cost_price|gender|total_amount|
+----------+-------+-------+--------+--------------+----------+----------------+----------------+----------+----------+----------+----------+-----------------+--------------------+---------+-----+-----+----+-------------+----------+------+------------+
|       403|   2270|    658|       1|          81.8|      81.8|             0.0|           0.00%|         0|      81.8|2025-06-16|App Mobile|       App Mobile| Elegant Satin Dress|  Dresses| Tiva|  Red|   L|         81.8|     45.12|Female|  

**total amount spend to purchase each product**

In [151]:
joined_df.groupBy("product_id").agg(sum("total_amount").alias("total_revenue")).orderBy(col("total_revenue").desc()).show()
joined_df.groupBy("product_id").agg(sum("total_amount").alias("total_revenue")).orderBy(col("total_revenue")).show()


+----------+------------------+
|product_id|     total_revenue|
+----------+------------------+
|       459|           1202.72|
|       116|           1189.48|
|       239|           1165.62|
|        19|           1135.52|
|        60|           1119.29|
|       310|           1110.34|
|       224|            1055.6|
|       440|            1041.6|
|       366|1032.6200000000003|
|        78|1030.2599999999998|
|       469|           1025.28|
|       288|1010.8500000000001|
|       219| 995.1999999999999|
|       410| 935.1600000000001|
|       307|            929.49|
|       445| 925.3599999999998|
|       297| 905.8600000000001|
|        63| 903.5200000000001|
|       276|            902.16|
|       280| 882.7999999999998|
+----------+------------------+
only showing top 20 rows
+----------+-----------------+
|product_id|    total_revenue|
+----------+-----------------+
|        64|            27.02|
|        66|            48.26|
|       187|            60.38|
|       346|         

**finding campagin duration**

In [152]:
fashion_store_campaigns_df.show()

+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|campaign_id|       campaign_name|start_date|  end_date|       channel|discount_type|discount_value|
+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|          1|   Spring Flash Sale|2025-04-01|2025-04-07|         Email|   Percentage|        10.00%|
|          2|    Easter Promotion|2025-04-08|2025-04-15|  Social Media|        Fixed|            10|
|          3|Mother's Day Camp...|2025-05-01|2025-05-09|         Email|   Percentage|        25.00%|
|          4|Mid-Season Clearance|2025-05-10|2025-05-19|    App Mobile|   Percentage|        30.00%|
|          5|           TIVA Week|2025-05-20|2025-05-31|  Social Media|   Percentage|        30.00%|
|          6|     June Price Drop|2025-06-01|2025-06-09|Website Banner|   Percentage|        10.00%|
|          7|  Early Summer Deals|2025-06-10|2025-06-17|         Email|        Fixed|      

In [153]:
from pyspark.sql.functions import datediff,col
fashion_store_campaigns_df.withColumn("campaign_duration", datediff("end_date","start_date")).show()

+-----------+--------------------+----------+----------+--------------+-------------+--------------+-----------------+
|campaign_id|       campaign_name|start_date|  end_date|       channel|discount_type|discount_value|campaign_duration|
+-----------+--------------------+----------+----------+--------------+-------------+--------------+-----------------+
|          1|   Spring Flash Sale|2025-04-01|2025-04-07|         Email|   Percentage|        10.00%|                6|
|          2|    Easter Promotion|2025-04-08|2025-04-15|  Social Media|        Fixed|            10|                7|
|          3|Mother's Day Camp...|2025-05-01|2025-05-09|         Email|   Percentage|        25.00%|                8|
|          4|Mid-Season Clearance|2025-05-10|2025-05-19|    App Mobile|   Percentage|        30.00%|                9|
|          5|           TIVA Week|2025-05-20|2025-05-31|  Social Media|   Percentage|        30.00%|               11|
|          6|     June Price Drop|2025-06-01|202

**working with date functions**

In [154]:
from pyspark.sql.functions import year,month,dayofmonth,dayofweek,dayofyear,months_between,abs

In [155]:
fashion_store_campaigns_df.withColumn("year",year("start_date"))\
                          .withColumn("month",month("start_date"))\
                          .withColumn("day",dayofmonth("start_date"))\
                          .withColumn("dayofweek",dayofweek("start_date"))\
                          .withColumn("dayofyear",dayofyear("start_date"))\
                          .withColumn("months_b/w_start_to_end",abs(months_between("start_date","end_date"))).show()

+-----------+--------------------+----------+----------+--------------+-------------+--------------+----+-----+---+---------+---------+-----------------------+
|campaign_id|       campaign_name|start_date|  end_date|       channel|discount_type|discount_value|year|month|day|dayofweek|dayofyear|months_b/w_start_to_end|
+-----------+--------------------+----------+----------+--------------+-------------+--------------+----+-----+---+---------+---------+-----------------------+
|          1|   Spring Flash Sale|2025-04-01|2025-04-07|         Email|   Percentage|        10.00%|2025|    4|  1|        3|       91|             0.19354839|
|          2|    Easter Promotion|2025-04-08|2025-04-15|  Social Media|        Fixed|            10|2025|    4|  8|        3|       98|             0.22580645|
|          3|Mother's Day Camp...|2025-05-01|2025-05-09|         Email|   Percentage|        25.00%|2025|    5|  1|        5|      121|             0.25806452|
|          4|Mid-Season Clearance|2025-0

In [156]:
fashion_store_campaigns_df.show()

+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|campaign_id|       campaign_name|start_date|  end_date|       channel|discount_type|discount_value|
+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|          1|   Spring Flash Sale|2025-04-01|2025-04-07|         Email|   Percentage|        10.00%|
|          2|    Easter Promotion|2025-04-08|2025-04-15|  Social Media|        Fixed|            10|
|          3|Mother's Day Camp...|2025-05-01|2025-05-09|         Email|   Percentage|        25.00%|
|          4|Mid-Season Clearance|2025-05-10|2025-05-19|    App Mobile|   Percentage|        30.00%|
|          5|           TIVA Week|2025-05-20|2025-05-31|  Social Media|   Percentage|        30.00%|
|          6|     June Price Drop|2025-06-01|2025-06-09|Website Banner|   Percentage|        10.00%|
|          7|  Early Summer Deals|2025-06-10|2025-06-17|         Email|        Fixed|      

**create new column with timestamp or try_to_timestamp**

In [157]:
from pyspark.sql.functions import timestamp_diff,col,try_to_timestamp,lit
# Define the expected timestamp format
timestamp_format = "yyyy-MM-dd HH:mm:ss"

fashion_store_campaigns_df.withColumn("timestamp_start_date",try_to_timestamp(col("start_date"),lit(timestamp_format)))\
                          .withColumn("timestamp_end_date",try_to_timestamp(col("end_date"),lit(timestamp_format)))\
                          .withColumn("timestamp_diff",abs(timestamp_diff("HOUR","timestamp_end_date","timestamp_start_date")))\
                          .show()

+-----------+--------------------+----------+----------+--------------+-------------+--------------+--------------------+-------------------+--------------+
|campaign_id|       campaign_name|start_date|  end_date|       channel|discount_type|discount_value|timestamp_start_date| timestamp_end_date|timestamp_diff|
+-----------+--------------------+----------+----------+--------------+-------------+--------------+--------------------+-------------------+--------------+
|          1|   Spring Flash Sale|2025-04-01|2025-04-07|         Email|   Percentage|        10.00%| 2025-04-01 00:00:00|2025-04-07 00:00:00|           144|
|          2|    Easter Promotion|2025-04-08|2025-04-15|  Social Media|        Fixed|            10| 2025-04-08 00:00:00|2025-04-15 00:00:00|           168|
|          3|Mother's Day Camp...|2025-05-01|2025-05-09|         Email|   Percentage|        25.00%| 2025-05-01 00:00:00|2025-05-09 00:00:00|           192|
|          4|Mid-Season Clearance|2025-05-10|2025-05-19|  

**Handling Missing Values**

**ways:**

**1. Drop missing Values ( for non - critical columns )**

**2. Fill missing values ( for numerical columns )**

**3. Impute Missing Values ( for continous data )**

**4. Drop missing Values ( for non - critical columns )**

**1. Drop missing Values ( for non - critical columns )**

In [158]:
fashion_store_campaigns_df.na.drop(how="any").show()

+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|campaign_id|       campaign_name|start_date|  end_date|       channel|discount_type|discount_value|
+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|          1|   Spring Flash Sale|2025-04-01|2025-04-07|         Email|   Percentage|        10.00%|
|          2|    Easter Promotion|2025-04-08|2025-04-15|  Social Media|        Fixed|            10|
|          3|Mother's Day Camp...|2025-05-01|2025-05-09|         Email|   Percentage|        25.00%|
|          4|Mid-Season Clearance|2025-05-10|2025-05-19|    App Mobile|   Percentage|        30.00%|
|          5|           TIVA Week|2025-05-20|2025-05-31|  Social Media|   Percentage|        30.00%|
|          6|     June Price Drop|2025-06-01|2025-06-09|Website Banner|   Percentage|        10.00%|
|          7|  Early Summer Deals|2025-06-10|2025-06-17|         Email|        Fixed|      

**2. Fill missing values ( for numerical columns )**

In [159]:
fashion_store_salesitems_df.printSchema()

root
 |-- item_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- channel: string (nullable = true)
 |-- channel_campaigns: string (nullable = true)



In [160]:
# way-1:
fashion_store_salesitems_df.fillna("2026-11-18",subset=["sale_date"]).show()
# way-2:
fashion_store_salesitems_df.fillna({"sale_date":"2026-11-18"}).show()

+-------+-------+----------+--------+--------------+----------+----------------+----------------+----------+----------+----------+----------+-----------------+
|item_id|sale_id|product_id|quantity|original_price|unit_price|discount_applied|discount_percent|discounted|item_total| sale_date|   channel|channel_campaigns|
+-------+-------+----------+--------+--------------+----------+----------------+----------------+----------+----------+----------+----------+-----------------+
|   2270|    658|       403|       1|          81.8|      81.8|             0.0|           0.00%|         0|      81.8|2025-06-16|App Mobile|       App Mobile|
|   1170|    336|       284|       1|         81.79|     81.79|             0.0|           0.00%|         0|     81.79|2025-06-17|E-commerce|   Website Banner|
|   2496|   1255|        71|       1|         80.76|     80.76|             0.0|           0.00%|         0|     80.76|2025-04-16|App Mobile|       App Mobile|
|   1273|    331|        98|       1|   

**3. Impute Missing Values ( for continous data )**

In [161]:
fashion_store_sales_df.printSchema()

root
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- country: string (nullable = true)



**implementing Imputer on total_amount column**

**fit() --> Fits a model to the input dataset with optional parameters**\
**transform() -->Transforms the input dataset with optional parameters.**

In [162]:
from pyspark.ml.feature import Imputer
# way-1:
Imputer_reference=Imputer(inputCols=["total_amount"],outputCols=["Imputed_total_amount"]).setStrategy("mean")
Imputer_Data_Cleaned_df1=Imputer_reference.fit(fashion_store_sales_df).transform(fashion_store_sales_df)

In [163]:
Imputer_Data_Cleaned_df1.show()

+-------+----------+----------+------------+----------+-----------+-----------+--------------------+
|sale_id|   channel|discounted|total_amount| sale_date|customer_id|    country|Imputed_total_amount|
+-------+----------+----------+------------+----------+-----------+-----------+--------------------+
|     10|E-commerce|         0|       299.7|2025-05-21|        195|     France|               299.7|
|    100|App Mobile|         0|      681.05|2025-04-21|        518|    Germany|              681.05|
|   1000|E-commerce|         0|       324.5|2025-05-20|        439|    Germany|               324.5|
|   1001|E-commerce|         0|      287.85|2025-04-05|        349|    Germany|              287.85|
|   1003|App Mobile|         0|      430.64|2025-06-06|        727|   Portugal|              430.64|
|   1004|App Mobile|         0|      104.73|2025-04-27|        983|    Germany|              104.73|
|   1006|E-commerce|         0|      249.21|2025-05-05|        309|      Spain|            

In [164]:
# way-2:
Imputer_reference=Imputer(inputCols=["total_amount"],outputCols=["Imputed_total_amount"],strategy='median')
Imputer_Data_Cleaned_df2=Imputer_reference.fit(fashion_store_sales_df).transform(fashion_store_sales_df)

In [165]:
Imputer_Data_Cleaned_df2.show()

+-------+----------+----------+------------+----------+-----------+-----------+--------------------+
|sale_id|   channel|discounted|total_amount| sale_date|customer_id|    country|Imputed_total_amount|
+-------+----------+----------+------------+----------+-----------+-----------+--------------------+
|     10|E-commerce|         0|       299.7|2025-05-21|        195|     France|               299.7|
|    100|App Mobile|         0|      681.05|2025-04-21|        518|    Germany|              681.05|
|   1000|E-commerce|         0|       324.5|2025-05-20|        439|    Germany|               324.5|
|   1001|E-commerce|         0|      287.85|2025-04-05|        349|    Germany|              287.85|
|   1003|App Mobile|         0|      430.64|2025-06-06|        727|   Portugal|              430.64|
|   1004|App Mobile|         0|      104.73|2025-04-27|        983|    Germany|              104.73|
|   1006|E-commerce|         0|      249.21|2025-05-05|        309|      Spain|            

**Verify Imputer functionality by making null records in dataframe**

In [166]:
fashion_store_sales_df_with_nulldata = fashion_store_sales_df.withColumn(
    "total_amount",
    when(
        (col("total_amount") != 287.85) &
        (col("total_amount") != 386.92) &
        (col("total_amount") != 175.37) &
        (col("total_amount") != 391.65),
        col("total_amount")
    ).otherwise(lit(None))
)

In [167]:
fashion_store_sales_df_with_nulldata.show()

+-------+----------+----------+------------+----------+-----------+-----------+
|sale_id|   channel|discounted|total_amount| sale_date|customer_id|    country|
+-------+----------+----------+------------+----------+-----------+-----------+
|     10|E-commerce|         0|       299.7|2025-05-21|        195|     France|
|    100|App Mobile|         0|      681.05|2025-04-21|        518|    Germany|
|   1000|E-commerce|         0|       324.5|2025-05-20|        439|    Germany|
|   1001|E-commerce|         0|        NULL|2025-04-05|        349|    Germany|
|   1003|App Mobile|         0|      430.64|2025-06-06|        727|   Portugal|
|   1004|App Mobile|         0|      104.73|2025-04-27|        983|    Germany|
|   1006|E-commerce|         0|      249.21|2025-05-05|        309|      Spain|
|   1007|App Mobile|         0|      437.52|2025-04-16|         53|      Italy|
|   1008|App Mobile|         0|        NULL|2025-05-24|        593|     France|
|   1011|App Mobile|         0|      212

In [168]:
Imputer_reference=Imputer(inputCols=["total_amount"],outputCols=["Imputed_total_amount"]).setStrategy("median")
Imputer_Data_Cleaned_df3=Imputer_reference.fit(fashion_store_sales_df_with_nulldata).transform(fashion_store_sales_df_with_nulldata)
Imputer_Data_Cleaned_df3.show()

mputer_reference=Imputer(inputCols=["total_amount"],outputCols=["Imputed_total_amount"]).setStrategy("mean")
Imputer_Data_Cleaned_df4=Imputer_reference.fit(fashion_store_sales_df_with_nulldata).transform(fashion_store_sales_df_with_nulldata)
Imputer_Data_Cleaned_df4.show()

mputer_reference=Imputer(inputCols=["total_amount"],outputCols=["Imputed_total_amount"]).setStrategy("mode")
Imputer_Data_Cleaned_df5=Imputer_reference.fit(fashion_store_sales_df_with_nulldata).transform(fashion_store_sales_df_with_nulldata)
Imputer_Data_Cleaned_df5.show()

mputer_reference=Imputer(inputCols=["total_amount"],outputCols=["Imputed_total_amount"]).setStrategy("standard_deviation")
Imputer_Data_Cleaned_df6=Imputer_reference.fit(fashion_store_sales_df_with_nulldata).transform(fashion_store_sales_df_with_nulldata)
Imputer_Data_Cleaned_df6.show()

+-------+----------+----------+------------+----------+-----------+-----------+--------------------+
|sale_id|   channel|discounted|total_amount| sale_date|customer_id|    country|Imputed_total_amount|
+-------+----------+----------+------------+----------+-----------+-----------+--------------------+
|     10|E-commerce|         0|       299.7|2025-05-21|        195|     France|               299.7|
|    100|App Mobile|         0|      681.05|2025-04-21|        518|    Germany|              681.05|
|   1000|E-commerce|         0|       324.5|2025-05-20|        439|    Germany|               324.5|
|   1001|E-commerce|         0|        NULL|2025-04-05|        349|    Germany|              343.05|
|   1003|App Mobile|         0|      430.64|2025-06-06|        727|   Portugal|              430.64|
|   1004|App Mobile|         0|      104.73|2025-04-27|        983|    Germany|              104.73|
|   1006|E-commerce|         0|      249.21|2025-05-05|        309|      Spain|            

**Drop Duplicates**

In [169]:
dropduplicates=fashion_store_salesitems_df.dropDuplicates()

In [170]:
dropduplicates.count()

2253

In [171]:
fashion_store_salesitems_df.count()

2253

### **Data Transformation**

In [172]:
fashion_store_campaigns_df.printSchema()
fashion_store_channels_df.printSchema()
fashion_store_customers_df.printSchema()
fashion_store_products_df.printSchema()
fashion_store_sales_df.printSchema()
fashion_store_salesitems_df.printSchema()
fashion_store_stock_df.printSchema()

root
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- channel: string (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)

root
 |-- channel: string (nullable = true)
 |-- description: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- age_range: string (nullable = true)
 |-- signup_date: date (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- color: string (nullable = true)
 |-- size: string (nullable = true)
 |-- catalog_price: double (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- gender: string (nullable = true)

root
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nul

In [173]:
fashion_store_campaigns_df.show()
fashion_store_channels_df.show()
fashion_store_customers_df.show()
fashion_store_products_df.show()
fashion_store_sales_df.show()
fashion_store_salesitems_df.show()
fashion_store_stock_df.show()

+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|campaign_id|       campaign_name|start_date|  end_date|       channel|discount_type|discount_value|
+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|          1|   Spring Flash Sale|2025-04-01|2025-04-07|         Email|   Percentage|        10.00%|
|          2|    Easter Promotion|2025-04-08|2025-04-15|  Social Media|        Fixed|            10|
|          3|Mother's Day Camp...|2025-05-01|2025-05-09|         Email|   Percentage|        25.00%|
|          4|Mid-Season Clearance|2025-05-10|2025-05-19|    App Mobile|   Percentage|        30.00%|
|          5|           TIVA Week|2025-05-20|2025-05-31|  Social Media|   Percentage|        30.00%|
|          6|     June Price Drop|2025-06-01|2025-06-09|Website Banner|   Percentage|        10.00%|
|          7|  Early Summer Deals|2025-06-10|2025-06-17|         Email|        Fixed|      

**joining fashion_store_channels_df**

In [174]:
joined_df=fashion_store_campaigns_df.join(fashion_store_channels_df,"channel","left")


In [175]:
joined_df.printSchema()
joined_df.show()

root
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)

+--------------+-----------+--------------------+----------+----------+-------------+--------------+----------------+
|       channel|campaign_id|       campaign_name|start_date|  end_date|discount_type|discount_value|     description|
+--------------+-----------+--------------------+----------+----------+-------------+--------------+----------------+
|         Email|          1|   Spring Flash Sale|2025-04-01|2025-04-07|   Percentage|        10.00%|            NULL|
|  Social Media|          2|    Easter Promotion|2025-04-08|2025-04-15|        Fixed|            10|            NULL|
|         Email|          3|Mother's Day Camp...|2025-05-01|2025

**joining fashion_store_sales_df**

In [176]:
joined_df=joined_df.join(fashion_store_sales_df,"channel")

In [177]:
joined_df.show()

+----------+-----------+--------------------+----------+----------+-------------+--------------+----------------+-------+----------+------------+----------+-----------+-----------+
|   channel|campaign_id|       campaign_name|start_date|  end_date|discount_type|discount_value|     description|sale_id|discounted|total_amount| sale_date|customer_id|    country|
+----------+-----------+--------------------+----------+----------+-------------+--------------+----------------+-------+----------+------------+----------+-----------+-----------+
|App Mobile|          4|Mid-Season Clearance|2025-05-10|2025-05-19|   Percentage|        30.00%|Brand mobile app|    994|         1|      477.09|2025-05-15|        798|   Portugal|
|App Mobile|          4|Mid-Season Clearance|2025-05-10|2025-05-19|   Percentage|        30.00%|Brand mobile app|    992|         1|      214.08|2025-05-13|        375|    Germany|
|App Mobile|          4|Mid-Season Clearance|2025-05-10|2025-05-19|   Percentage|        30.00%

In [178]:
joined_df.printSchema()

root
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- country: string (nullable = true)



**joining fashion_store_salesitems_df**

In [179]:
joined_df=joined_df.join(fashion_store_salesitems_df,"sale_id")

In [180]:
joined_df.printSchema()
joined_df.show()

root
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_dat

**joining fashion_store_products_df**

In [181]:
joined_df=joined_df.join(fashion_store_products_df,"product_id")

In [182]:
joined_df.printSchema( )
joined_df.show()

root
 |-- product_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_dat

**joining fashion_store_customers_df**

In [183]:
joined_df=joined_df.join(fashion_store_customers_df,"customer_id")

In [184]:
joined_df.printSchema()
joined_df.show()

root
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_dat

**joining fashion_store_stock_df**

In [185]:
joined_df=joined_df.join(fashion_store_stock_df,"product_id")

In [186]:
joined_df.printSchema()
joined_df.show()

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_dat

In [187]:
# To fix duplicate/ambiguous columns like 'country', we can drop them or rename them.
# Since 'country' is present in several of our original DataFrames (sales, customers, stock),
# we can drop the duplicates by referencing the specific DataFrame from which they came.

# Let's inspect the columns first to see all references, then resolve the ambiguity:
# A clean way to avoid this during joining is to drop 'country' from the right-side DataFrame
# before performing the join if you only need the country from the primary DataFrame (e.g., sales).

# Let's recreate a clean joined_df by dropping redundant 'country' columns before joining them:
cleaned_joined_df = fashion_store_campaigns_df.join(fashion_store_channels_df, "channel", "left") \
    .join(fashion_store_sales_df, "channel") \
    .join(fashion_store_salesitems_df, "sale_id") \
    .join(fashion_store_products_df, "product_id") \
    .join(fashion_store_customers_df.drop("country"), "customer_id") \
    .join(fashion_store_stock_df.drop("country"), "product_id")

# Now, cleaned_joined_df will have only one 'country' column (the one from fashion_store_sales_df)
cleaned_joined_df.printSchema()
cleaned_joined_df.show(5)

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_dat

In [188]:
joined_df.groupBy("product_id").agg(sum("total_amount").alias("total_revenue")).show()

+----------+------------------+
|product_id|     total_revenue|
+----------+------------------+
|       496|           2600.58|
|       471|           2408.52|
|       148|1177.5600000000002|
|       463|1449.2800000000002|
|       243|            365.96|
|       392|           2899.76|
|        31|            2600.5|
|       451|           1388.48|
|       251|1289.3200000000002|
|        85|            824.26|
|       137|            2018.5|
|        65|2047.8799999999999|
|       458|            1435.9|
|       481|           3035.96|
|        53|2446.8999999999996|
|       255|2298.9599999999996|
|       133|3282.7199999999993|
|       296|           4098.12|
|       472|            1571.7|
|        78|           5384.54|
+----------+------------------+
only showing top 20 rows


### **Handling Duplicate 'Country' Columns by Renaming Them**

Instead of losing the country information from the customers or stock DataFrames, we will rename each before joining so they all exist without conflicts (e.g., `sale_country`, `customer_country`, `stock_country`). Then we can consolidate them into a single master country column.

In [189]:
from pyspark.sql.functions import coalesce

# 1. Prepare original dataframes by renaming the 'country' column to match their context
sales_df_prepared = fashion_store_sales_df.withColumnRenamed("country", "sale_country")
customers_df_prepared = fashion_store_customers_df.withColumnRenamed("country", "customer_country")
stock_df_prepared = fashion_store_stock_df.withColumnRenamed("country", "stock_country")

# 2. Perform the join sequence with the prepared DataFrames
explicit_joined_df = fashion_store_campaigns_df.join(fashion_store_channels_df, "channel", "left") \
    .join(sales_df_prepared, "channel") \
    .join(fashion_store_salesitems_df, "sale_id") \
    .join(fashion_store_products_df, "product_id") \
    .join(customers_df_prepared, "customer_id") \
    .join(stock_df_prepared, "product_id")

# 3. Create a single consolidated 'country' column by prioritizing the sale country,
# then falling back to customer country, then stock country if any nulls exist.
final_df = explicit_joined_df.withColumn(
    "country",
    coalesce("sale_country", "customer_country", "stock_country")
)

# 4. If you no longer need the separate source country columns, you can drop them now safely
# because their unique data has been merged/coalesced into 'country'.
final_df = final_df.drop("sale_country", "customer_country", "stock_country")

# Verify schema and check data
final_df.printSchema()
final_df.select("product_id", "customer_id", "sale_id", "country").show(10)

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- channel:

In [190]:
final_df.show()

+----------+-----------+-------+----------+-----------+--------------------+----------+----------+-------------+--------------+----------------+----------+------------+----------+-------+--------+--------------+----------+----------------+----------------+----------+----------+----------+----------+-----------------+--------------------+---------+-----+-----+----+-------------+----------+------+---------+-----------+--------------+-----------+
|product_id|customer_id|sale_id|   channel|campaign_id|       campaign_name|start_date|  end_date|discount_type|discount_value|     description|discounted|total_amount| sale_date|item_id|quantity|original_price|unit_price|discount_applied|discount_percent|discounted|item_total| sale_date|   channel|channel_campaigns|        product_name| category|brand|color|size|catalog_price|cost_price|gender|age_range|signup_date|stock_quantity|    country|
+----------+-----------+-------+----------+-----------+--------------------+----------+----------+------

In [191]:
final_df.groupBy("country").count().show()

+-----------+-----+
|    country|count|
+-----------+-----+
|    Germany|  542|
|     France|  478|
|      Italy|  392|
|      Spain|  238|
|   Portugal|  230|
|Netherlands|  286|
+-----------+-----+



In [192]:
final_df.groupBy("country").agg(count("product_id").alias("product_count")).show()


+-----------+-------------+
|    country|product_count|
+-----------+-------------+
|    Germany|          542|
|     France|          478|
|      Italy|          392|
|      Spain|          238|
|   Portugal|          230|
|Netherlands|          286|
+-----------+-------------+



### **Working with Advanced Transformation**


**pyspark.sql.DataFrame.approxQuantile DataFrame.approxQuantile(col, probabilities, relativeError)[source]**

In [193]:


total_amount_quantile_data=final_df.approxQuantile("total_amount",[0.1,0.99],0.00)
original_price_quantile_data=final_df.approxQuantile("original_price",[0.1,0.99],0.00)
unit_price_quantile_data=final_df.approxQuantile("unit_price",[0.1,0.99],0.00)

In [194]:
total_amount_quantile_data

[178.9, 747.5]

In [195]:
low_cutoff_total_amount=total_amount_quantile_data[0]
high_cutoff_total_amount=total_amount_quantile_data[1]

low_cutoff_total_amount,high_cutoff_total_amount

(178.9, 747.5)

In [196]:
original_price_quantile_data

[32.94, 80.76]

In [197]:
low_cutoff_original_price=original_price_quantile_data[0]
high_cutoff_original_price=original_price_quantile_data[1]

low_cutoff_original_price,high_cutoff_original_price

(32.94, 80.76)

In [198]:
unit_price_quantile_data

[30.55, 79.7]

In [199]:
low_cutoff_unit_price=unit_price_quantile_data[0]
high_cutoff_unit_price=unit_price_quantile_data[1]

low_cutoff_unit_price,high_cutoff_unit_price

(30.55, 79.7)

**statistics functions**

In [200]:
final_df.select("total_amount","original_price","unit_price").summary().show()

+-------+------------------+-----------------+------------------+
|summary|      total_amount|   original_price|        unit_price|
+-------+------------------+-----------------+------------------+
|  count|              2166|             2166|              2166|
|   mean|367.79854108956556| 49.8179039704522|47.636500461680264|
| stddev|153.02199489745658|12.86952183567811|13.470038384337268|
|    min|              45.5|            14.64|             11.68|
|    25%|            245.94|            40.25|             38.07|
|    50%|            357.92|            50.55|             47.44|
|    75%|            467.38|             58.7|             57.31|
|    max|            859.38|             81.8|              81.8|
+-------+------------------+-----------------+------------------+



In [201]:
final_df.select("total_amount","original_price","unit_price").describe().show()

+-------+------------------+-----------------+------------------+
|summary|      total_amount|   original_price|        unit_price|
+-------+------------------+-----------------+------------------+
|  count|              2166|             2166|              2166|
|   mean|367.79854108956556| 49.8179039704522|47.636500461680264|
| stddev|153.02199489745658|12.86952183567811|13.470038384337268|
|    min|              45.5|            14.64|             11.68|
|    max|            859.38|             81.8|              81.8|
+-------+------------------+-----------------+------------------+



In [202]:
final_df.filter((col("total_amount")>low_cutoff_total_amount )& (col("total_amount")<high_cutoff_total_amount)).show()

+----------+-----------+-------+----------+-----------+--------------------+----------+----------+-------------+--------------+----------------+----------+------------+----------+-------+--------+--------------+----------+----------------+----------------+----------+----------+----------+----------+-----------------+--------------------+---------+-----+-----+----+-------------+----------+------+---------+-----------+--------------+-----------+
|product_id|customer_id|sale_id|   channel|campaign_id|       campaign_name|start_date|  end_date|discount_type|discount_value|     description|discounted|total_amount| sale_date|item_id|quantity|original_price|unit_price|discount_applied|discount_percent|discounted|item_total| sale_date|   channel|channel_campaigns|        product_name| category|brand|color|size|catalog_price|cost_price|gender|age_range|signup_date|stock_quantity|    country|
+----------+-----------+-------+----------+-----------+--------------------+----------+----------+------

In [203]:
final_df.filter((col("total_amount")>=low_cutoff_total_amount )& (col("total_amount")<=high_cutoff_total_amount)).show()

+----------+-----------+-------+----------+-----------+--------------------+----------+----------+-------------+--------------+----------------+----------+------------+----------+-------+--------+--------------+----------+----------------+----------------+----------+----------+----------+----------+-----------------+--------------------+---------+-----+-----+----+-------------+----------+------+---------+-----------+--------------+-----------+
|product_id|customer_id|sale_id|   channel|campaign_id|       campaign_name|start_date|  end_date|discount_type|discount_value|     description|discounted|total_amount| sale_date|item_id|quantity|original_price|unit_price|discount_applied|discount_percent|discounted|item_total| sale_date|   channel|channel_campaigns|        product_name| category|brand|color|size|catalog_price|cost_price|gender|age_range|signup_date|stock_quantity|    country|
+----------+-----------+-------+----------+-----------+--------------------+----------+----------+------

In [204]:
from pyspark.sql.functions import min, max

# Correct way to aggregate both min and max simultaneously
final_df.agg(
    min("item_total").alias("min_item_total"),
    max("item_total").alias("max_item_total")
).show()

+--------------+--------------+
|min_item_total|max_item_total|
+--------------+--------------+
|         15.07|         403.8|
+--------------+--------------+



**create new column items_stock_package_weight using item_total column**

In [205]:
final_df.withColumn("items_stock_package_weight",when(col("item_total")<=100,"low_package")\
                    .when((col("item_total")>100) & (col("item_total")<=300),"medium_package")\
                    .otherwise("high_package")).show()

+----------+-----------+-------+----------+-----------+--------------------+----------+----------+-------------+--------------+----------------+----------+------------+----------+-------+--------+--------------+----------+----------------+----------------+----------+----------+----------+----------+-----------------+--------------------+---------+-----+-----+----+-------------+----------+------+---------+-----------+--------------+-----------+--------------------------+
|product_id|customer_id|sale_id|   channel|campaign_id|       campaign_name|start_date|  end_date|discount_type|discount_value|     description|discounted|total_amount| sale_date|item_id|quantity|original_price|unit_price|discount_applied|discount_percent|discounted|item_total| sale_date|   channel|channel_campaigns|        product_name| category|brand|color|size|catalog_price|cost_price|gender|age_range|signup_date|stock_quantity|    country|items_stock_package_weight|
+----------+-----------+-------+----------+-------

In [206]:
final_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- channel:

**writing dataframe as parquet files**

**Removing duplicate columns**

In [207]:
from pyspark.sql.functions import col

# Create a list of unique column names by appending an index to any duplicate columns
unique_cols = []
col_counts = {}
for name in final_df.columns:
    if name in col_counts:
        col_counts[name] += 1
        unique_cols.append(f"{name}_{col_counts[name]}")
    else:
        col_counts[name] = 0
        unique_cols.append(name)

# Rename the duplicate columns in final_df using the unique column names
final_df_unique = final_df.toDF(*unique_cols)

# Write the deduplicated DataFrame to Parquet
final_df_unique.write.option("header", "true").mode("overwrite").parquet("/content/parquet_files/fashion_store_joined_dataframe/")

###**Advanced and few more Transformations and aggrigations**

In [208]:
final_df_unique.printSchema()
final_df_unique.show()

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted_1: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date_1: date (nullable = true)
 |-- chan

**working  with Window functions**

**Ranking to selling products bought by customer using Window function**

In [209]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank,dense_rank
window_spec=Window.partitionBy("product_id").orderBy(col("customer_id").desc())


final_df_unique.withColumn("rank_fun",rank().over(window_spec))\
               .withColumn("dense_rank_fun",dense_rank()
               .over(window_spec)).show()

+----------+-----------+-------+----------+-----------+--------------------+----------+----------+-------------+--------------+----------------+----------+------------+----------+-------+--------+--------------+----------+----------------+----------------+------------+----------+-----------+----------+-----------------+------------------+--------+-----+-----+----+-------------+----------+------+---------+-----------+--------------+-------+--------+--------------+
|product_id|customer_id|sale_id|   channel|campaign_id|       campaign_name|start_date|  end_date|discount_type|discount_value|     description|discounted|total_amount| sale_date|item_id|quantity|original_price|unit_price|discount_applied|discount_percent|discounted_1|item_total|sale_date_1| channel_1|channel_campaigns|      product_name|category|brand|color|size|catalog_price|cost_price|gender|age_range|signup_date|stock_quantity|country|rank_fun|dense_rank_fun|
+----------+-----------+-------+----------+-----------+---------

**Aggrigation**

**Total products buy by customer_id**

In [210]:
final_df.groupBy("customer_id").agg(count("product_id").alias("total_products_buy")).show()

+-----------+------------------+
|customer_id|total_products_buy|
+-----------+------------------+
|        496|                 4|
|        833|                 6|
|        897|                 4|
|        392|                 4|
|        451|                 6|
|        580|                 4|
|        883|                 4|
|        879|                 6|
|        898|                10|
|        804|                 6|
|        588|                 6|
|        481|                 6|
|         53|                 4|
|        133|                 4|
|        296|                 4|
|        513|                 4|
|        613|                16|
|        961|                 6|
|        857|                 4|
|        974|                 4|
+-----------+------------------+
only showing top 20 rows


**Top 10 products bought by customer**

In [211]:
final_df.groupBy("customer_id").agg(count("product_id").alias("total_products_buy")).orderBy(col("total_products_buy").desc()).show(10)


final_df.groupBy("customer_id")\
        .agg(count("product_id").alias("total_products_buy"))\
        .orderBy(col("total_products_buy").desc()).limit(10).show()


+-----------+------------------+
|customer_id|total_products_buy|
+-----------+------------------+
|        220|                20|
|        613|                16|
|        497|                16|
|        346|                14|
|        727|                14|
|        233|                14|
|        277|                14|
|        854|                12|
|        675|                12|
|         86|                12|
+-----------+------------------+
only showing top 10 rows
+-----------+------------------+
|customer_id|total_products_buy|
+-----------+------------------+
|        220|                20|
|        497|                16|
|        613|                16|
|        233|                14|
|        727|                14|
|        346|                14|
|        277|                14|
|         86|                12|
|        191|                12|
|        398|                12|
+-----------+------------------+



**Top 10 customers based on Spending**

In [212]:
final_df.groupBy("customer_id").agg(sum("total_amount").alias("total_spending")).orderBy(col("total_spending").desc()).show(10)


+-----------+------------------+
|customer_id|    total_spending|
+-----------+------------------+
|        220|10972.059999999998|
|        613| 6927.300000000002|
|        720| 6741.120000000002|
|        253|           6374.12|
|        191| 6277.739999999999|
|        727| 6111.680000000001|
|        675| 5931.299999999999|
|        854| 5866.140000000001|
|        277| 5587.339999999998|
|        750| 5436.059999999999|
+-----------+------------------+
only showing top 10 rows


**Total Revenue and Total Products bought by customer**

In [213]:
final_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- channel:

In [214]:
from pyspark.sql.functions import count, sum, col, avg, round

final_df.groupBy("customer_id").agg(
    count("product_id").alias("total_purchases"),
    sum(col("total_amount")).alias("total_revenue"),
    round(avg("total_amount"), 2).alias("average_value")
).orderBy(col("total_revenue").desc()).show()

+-----------+---------------+------------------+-------------+
|customer_id|total_purchases|     total_revenue|average_value|
+-----------+---------------+------------------+-------------+
|        220|             20|10972.059999999998|        548.6|
|        613|             16| 6927.300000000002|       432.96|
|        720|             12| 6741.120000000002|       561.76|
|        253|             10|           6374.12|       637.41|
|        191|             12| 6277.739999999999|       523.14|
|        727|             14| 6111.680000000001|       436.55|
|        675|             12| 5931.299999999999|       494.27|
|        854|             12| 5866.140000000001|       488.85|
|        277|             14| 5587.339999999998|        399.1|
|        750|             12| 5436.059999999999|        453.0|
|        682|             12| 5410.140000000001|       450.85|
|        795|             12| 5310.899999999999|       442.57|
|        346|             14| 5259.419999999999|       

In [215]:
from pyspark.sql.functions import count, sum, col, avg, round,stddev,max,min,mean
final_df.groupBy("customer_id").agg(
    count("product_id").alias("total_purchases"),
    sum(col("total_amount")).alias("total_revenue"),
    round(avg("total_amount"), 2).alias("average_value"),
    round(stddev("total_amount"), 2).alias("std_deviation_total_amount"),
    round(max("total_amount"), 2).alias("max_value_total_amount"),
    round(min("total_amount"), 2).alias("min_value_total_amount"),
    round(mean("total_amount"), 2).alias("mean_value_total_amount")
).orderBy(col("total_revenue").desc()).show()

+-----------+---------------+------------------+-------------+--------------------------+----------------------+----------------------+-----------------------+
|customer_id|total_purchases|     total_revenue|average_value|std_deviation_total_amount|max_value_total_amount|min_value_total_amount|mean_value_total_amount|
+-----------+---------------+------------------+-------------+--------------------------+----------------------+----------------------+-----------------------+
|        220|             20|10972.059999999998|        548.6|                    121.93|                699.78|                425.23|                  548.6|
|        613|             16| 6927.300000000002|       432.96|                    190.69|                670.95|                274.38|                 432.96|
|        720|             12| 6741.120000000002|       561.76|                    160.45|                715.38|                408.14|                 561.76|
|        253|             10|           

**Product Popularity Metrics -- round(),avg(),stddev(),collect_set(),collect_list()**

In [216]:
from pyspark.sql.functions import collect_list,collect_set
final_df.groupBy("product_id").agg( collect_set("sale_id").alias("unique_sellers"),
    collect_list("sale_id").alias("all_sellers_with_duplicates")).show()

+----------+--------------------+---------------------------+
|product_id|      unique_sellers|all_sellers_with_duplicates|
+----------+--------------------+---------------------------+
|         1|         [1049, 476]|       [476, 476, 1049, ...|
|         2|          [281, 713]|       [713, 713, 281, 281]|
|         3|          [180, 152]|       [180, 180, 152, 152]|
|         5|[56, 411, 1132, 799]|       [799, 799, 56, 56...|
|         6|                [32]|                   [32, 32]|
|         8|         [694, 1015]|       [694, 694, 1015, ...|
|         9|               [557]|                 [557, 557]|
|        10|          [240, 505]|       [505, 505, 240, 240]|
|        11|    [552, 1070, 127]|       [552, 552, 127, 1...|
|        12|      [161, 497, 46]|       [497, 497, 46, 46...|
|        13|            [3, 286]|           [3, 3, 286, 286]|
|        14|    [1313, 588, 399]|       [588, 588, 399, 3...|
|        15|          [190, 767]|       [767, 767, 190, 190]|
|       

**Monthly Revenue and order count Trend ---- Analysis**

step-1: extract "month" from order_purchase_timestamp\
step-2: code for finding\
total_orders,\
total_revenue,\
avg_order_value,\
min_order_value,\
max_order_value

In [217]:
from pyspark.sql.functions import month, count, sum, avg, min, max, round, desc

# Using final_df_unique resolves the ambiguous column conflict
final_df_unique.withColumn("month", month("sale_date")).groupBy("month")\
.agg(
     count("product_id").alias("total_orders"),
     sum("total_amount").alias("total_revenue"),
     round(avg("total_amount"), 2).alias("avg_order_value"),
     min("total_amount").alias("min_order_value"),
     max("total_amount").alias("max_order_value")
)\
.orderBy(desc("total_orders")).show()

+-----+------------+------------------+---------------+---------------+---------------+
|month|total_orders|     total_revenue|avg_order_value|min_order_value|max_order_value|
+-----+------------+------------------+---------------+---------------+---------------+
|    5|         966|341897.65999999957|         353.93|           45.5|         859.38|
|    4|         884| 335841.2799999996|         379.91|          88.35|          760.4|
|    6|         316|118912.70000000029|         376.31|          88.11|         712.86|
+-----+------------+------------------+---------------+---------------+---------------+



**working with first() and last()**

In [218]:
final_df_unique.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted_1: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date_1: date (nullable = true)
 |-- chan

In [219]:
from pyspark.sql.functions  import first,last,any_value,timestamp_diff
final_df_unique.groupBy("customer_id")\
.agg(
    first("sale_date").alias("first_purchase_sale_date"),
    last("sale_date").alias("last_order_sale_date"),
    timestamp_diff("HOUR",any_value("start_date"),any_value("end_date")).alias("hours_since_first_purchase"),
    count("product_id").alias("total_products_purchased"),
    avg("total_amount").alias("avg_purchase_value")

)\
.orderBy(desc("total_products_purchased")).show()

+-----------+------------------------+--------------------+--------------------------+------------------------+------------------+
|customer_id|first_purchase_sale_date|last_order_sale_date|hours_since_first_purchase|total_products_purchased|avg_purchase_value|
+-----------+------------------------+--------------------+--------------------------+------------------------+------------------+
|        220|              2025-05-27|          2025-06-05|                       216|                      20| 548.6029999999998|
|        613|              2025-05-20|          2025-05-22|                       216|                      16| 432.9562500000001|
|        497|              2025-06-13|          2025-04-23|                       216|                      16| 303.9962500000001|
|        727|              2025-04-19|          2025-06-06|                       216|                      14| 436.5485714285715|
|        233|              2025-06-15|          2025-04-27|                       2

**working with concat()**

In [220]:
from pyspark.sql.functions import concat

final_df_unique.withColumn("product_specifications",concat("category","brand","size")).show()

+----------+-----------+-------+----------+-----------+--------------------+----------+----------+-------------+--------------+----------------+----------+------------+----------+-------+--------+--------------+----------+----------------+----------------+------------+----------+-----------+----------+-----------------+--------------------+---------+-----+-----+----+-------------+----------+------+---------+-----------+--------------+-----------+----------------------+
|product_id|customer_id|sale_id|   channel|campaign_id|       campaign_name|start_date|  end_date|discount_type|discount_value|     description|discounted|total_amount| sale_date|item_id|quantity|original_price|unit_price|discount_applied|discount_percent|discounted_1|item_total|sale_date_1| channel_1|channel_campaigns|        product_name| category|brand|color|size|catalog_price|cost_price|gender|age_range|signup_date|stock_quantity|    country|product_specifications|
+----------+-----------+-------+----------+---------

**To concatenate column data with a delimiter like a comma (,), you can use the PySpark SQL function concat_ws (Concatenate with Separator).**

In [221]:
from pyspark.sql.functions import concat_ws

# concat_ws takes the delimiter as the first argument, followed by the columns to concatenate
final_df_unique.withColumn(
    "product_specifications",
    concat_ws(", ", "category", "brand", "size")
).show()

+----------+-----------+-------+----------+-----------+--------------------+----------+----------+-------------+--------------+----------------+----------+------------+----------+-------+--------+--------------+----------+----------------+----------------+------------+----------+-----------+----------+-----------------+--------------------+---------+-----+-----+----+-------------+----------+------+---------+-----------+--------------+-----------+----------------------+
|product_id|customer_id|sale_id|   channel|campaign_id|       campaign_name|start_date|  end_date|discount_type|discount_value|     description|discounted|total_amount| sale_date|item_id|quantity|original_price|unit_price|discount_applied|discount_percent|discounted_1|item_total|sale_date_1| channel_1|channel_campaigns|        product_name| category|brand|color|size|catalog_price|cost_price|gender|age_range|signup_date|stock_quantity|    country|product_specifications|
+----------+-----------+-------+----------+---------

### **Performance/Property Optimization**

In [222]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Performance/Property Optimization").getOrCreate()

In [223]:
fashion_store_campaigns_csv_path="/content/kaggleresources/dataset_fashion_store_campaigns.csv"
fashion_store_channels_csv_path="/content/kaggleresources/dataset_fashion_store_channels.csv"
fashion_store_customers_csv_path="/content/kaggleresources/dataset_fashion_store_customers.csv"
fashion_store_products_csv_path="/content/kaggleresources/dataset_fashion_store_products.csv"
fashion_store_sales_csv_path="/content/kaggleresources/dataset_fashion_store_sales.csv"
fashion_store_salesitems_csv_path="/content/kaggleresources/dataset_fashion_store_salesitems.csv"
fashion_store_stock_csv_path="/content/kaggleresources/dataset_fashion_store_stock.csv"

In [224]:
fashion_store_campaigns_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_campaigns_csv_path)
fashion_store_channels_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_channels_csv_path)
fashion_store_customers_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_customers_csv_path)
fashion_store_products_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_products_csv_path)
fashion_store_sales_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_sales_csv_path)
fashion_store_salesitems_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_salesitems_csv_path)
fashion_store_stock_df=spark.read.option("header","true").option("inferschema","true").csv(fashion_store_stock_csv_path)

In [225]:
fashion_store_campaigns_df.show()
fashion_store_channels_df.show()
fashion_store_customers_df.show()
fashion_store_products_df.show()
fashion_store_sales_df.show()
fashion_store_salesitems_df.show()
fashion_store_stock_df.show()

+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|campaign_id|       campaign_name|start_date|  end_date|       channel|discount_type|discount_value|
+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|          1|   Spring Flash Sale|2025-04-01|2025-04-07|         Email|   Percentage|        10.00%|
|          2|    Easter Promotion|2025-04-08|2025-04-15|  Social Media|        Fixed|            10|
|          3|Mother's Day Camp...|2025-05-01|2025-05-09|         Email|   Percentage|        25.00%|
|          4|Mid-Season Clearance|2025-05-10|2025-05-19|    App Mobile|   Percentage|        30.00%|
|          5|           TIVA Week|2025-05-20|2025-05-31|  Social Media|   Percentage|        30.00%|
|          6|     June Price Drop|2025-06-01|2025-06-09|Website Banner|   Percentage|        10.00%|
|          7|  Early Summer Deals|2025-06-10|2025-06-17|         Email|        Fixed|      

**Implementing cache using cache()**

In [226]:
fashion_store_campaigns_df.cache()
fashion_store_channels_df.cache()
fashion_store_customers_df.cache()
fashion_store_products_df.cache()
fashion_store_sales_df.cache()
fashion_store_salesitems_df.cache()
fashion_store_stock_df.cache()

DataFrame[country: string, product_id: int, stock_quantity: int]

**Optimized join strategies**

In [227]:
from pyspark.sql.functions import broadcast

broadcast_df=broadcast(fashion_store_campaigns_df)
broadcast_df.show()

+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|campaign_id|       campaign_name|start_date|  end_date|       channel|discount_type|discount_value|
+-----------+--------------------+----------+----------+--------------+-------------+--------------+
|          1|   Spring Flash Sale|2025-04-01|2025-04-07|         Email|   Percentage|        10.00%|
|          2|    Easter Promotion|2025-04-08|2025-04-15|  Social Media|        Fixed|            10|
|          3|Mother's Day Camp...|2025-05-01|2025-05-09|         Email|   Percentage|        25.00%|
|          4|Mid-Season Clearance|2025-05-10|2025-05-19|    App Mobile|   Percentage|        30.00%|
|          5|           TIVA Week|2025-05-20|2025-05-31|  Social Media|   Percentage|        30.00%|
|          6|     June Price Drop|2025-06-01|2025-06-09|Website Banner|   Percentage|        10.00%|
|          7|  Early Summer Deals|2025-06-10|2025-06-17|         Email|        Fixed|      

In [233]:
from pyspark.sql.functions import coalesce

# 1. Prepare original dataframes by renaming the 'country' column to match their context
sales_df_prepared = fashion_store_sales_df.withColumnRenamed("country", "sale_country")
customers_df_prepared = fashion_store_customers_df.withColumnRenamed("country", "customer_country")
stock_df_prepared = fashion_store_stock_df.withColumnRenamed("country", "stock_country")

# 2. Perform the join sequence with the prepared DataFrames
explicit_joined_df = fashion_store_campaigns_df.join(fashion_store_channels_df, "channel", "left") \
    .join(sales_df_prepared, "channel") \
    .join(fashion_store_salesitems_df, "sale_id") \
    .join(fashion_store_products_df, "product_id") \
    .join(customers_df_prepared, "customer_id") \
    .join(stock_df_prepared, "product_id")

# 3. Create a single consolidated 'country' column by prioritizing the sale country,
# then falling back to customer country, then stock country if any nulls exist.
final_df = explicit_joined_df.withColumn(
    "country",
    coalesce("sale_country", "customer_country", "stock_country")
)

# 4. If you no longer need the separate source country columns, you can drop them now safely
# because their unique data has been merged/coalesced into 'country'.
final_df = final_df.drop("sale_country", "customer_country", "stock_country")

# Verify schema and check data
final_df.printSchema()
final_df.select("product_id", "customer_id", "sale_id", "country").show()

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- channel:

In [234]:
final_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- channel:

In [235]:
from pyspark.sql.functions import col

# Create a list of unique column names by appending an index to any duplicate columns
unique_cols = []
col_counts = {}
for name in final_df.columns:
    if name in col_counts:
        col_counts[name] += 1
        unique_cols.append(f"{name}_{col_counts[name]}")
    else:
        col_counts[name] = 0
        unique_cols.append(name)

# Rename the duplicate columns in final_df using the unique column names
final_df_unique = final_df.toDF(*unique_cols)


In [236]:
final_df_unique.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted_1: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date_1: date (nullable = true)
 |-- chan

In [237]:
final_df_unique.show()

+----------+-----------+-------+----------+-----------+--------------------+----------+----------+-------------+--------------+----------------+----------+------------+----------+-------+--------+--------------+----------+----------------+----------------+------------+----------+-----------+----------+-----------------+--------------------+---------+-----+-----+----+-------------+----------+------+---------+-----------+--------------+-----------+
|product_id|customer_id|sale_id|   channel|campaign_id|       campaign_name|start_date|  end_date|discount_type|discount_value|     description|discounted|total_amount| sale_date|item_id|quantity|original_price|unit_price|discount_applied|discount_percent|discounted_1|item_total|sale_date_1| channel_1|channel_campaigns|        product_name| category|brand|color|size|catalog_price|cost_price|gender|age_range|signup_date|stock_quantity|    country|
+----------+-----------+-------+----------+-----------+--------------------+----------+----------+

**1.BroadCast Join ---- broadcast()**\
Marks a DataFrame as small enough for use in broadcast joins.

Returns\
~pyspark.sql.DataFrame
    DataFrame marked as ready for broadcast join.

In [239]:
broadcast_join_df=broadcast(fashion_store_products_df)
broadcast_result=final_df_unique.join(broadcast_join_df,"product_id")
broadcast_result.show()

+----------+-----------+-------+----------+-----------+--------------------+----------+----------+-------------+--------------+----------------+----------+------------+----------+-------+--------+--------------+----------+----------------+----------------+------------+----------+-----------+----------+-----------------+--------------------+---------+-----+-----+----+-------------+----------+------+---------+-----------+--------------+-----------+--------------------+---------+-----+-----+----+-------------+----------+------+
|product_id|customer_id|sale_id|   channel|campaign_id|       campaign_name|start_date|  end_date|discount_type|discount_value|     description|discounted|total_amount| sale_date|item_id|quantity|original_price|unit_price|discount_applied|discount_percent|discounted_1|item_total|sale_date_1| channel_1|channel_campaigns|        product_name| category|brand|color|size|catalog_price|cost_price|gender|age_range|signup_date|stock_quantity|    country|        product_nam

In [240]:
broadcast_result.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted_1: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date_1: date (nullable = true)
 |-- chan

**2.sort & merge**\
def **sortWithinPartitions**(
    *cols: int | str | Column | List[int | str | Column],
    **kwargs: Any
) -> DataFrame

**Returns a new DataFrame with each partition sorted by the specified column(s).**

Parameters

cols : int, str, list or Column, optional
    list of Column or column names or column ordinals to sort by.

In [245]:
repartitioned_fashion_store_products_df=fashion_store_products_df.sortWithinPartitions("product_id")
repartitioned_fashion_store_sales_df=fashion_store_sales_df.sortWithinPartitions("sale_id")
repartitioned_fashion_store_salesitems_df=fashion_store_salesitems_df.sortWithinPartitions("sale_id")

In [246]:
repartitioned_fashion_store_products_df.show()
repartitioned_fashion_store_sales_df.show()
repartitioned_fashion_store_salesitems_df.show()

+----------+--------------------+--------+-----+-----+----+-------------+----------+------+
|product_id|        product_name|category|brand|color|size|catalog_price|cost_price|gender|
+----------+--------------------+--------+-----+-----+----+-------------+----------+------+
|         1|     Soft Wrap Dress| Dresses| Tiva|Green|   S|        40.41|      20.7|Female|
|         2|       Soft Wrap Tee|T-Shirts| Tiva|White|   S|        78.45|     53.76|Female|
|         3|      Soft Linen Tee|T-Shirts| Tiva|Green|  XL|         23.9|     14.81|Female|
|         4|     Soft Ribbed Tee|T-Shirts| Tiva|White|   S|         60.0|     34.78|Female|
|         5|  Soft Wrap Trousers|   Pants| Tiva| Blue|   M|        36.84|     16.46|Female|
|         6|     Soft Wrap Shoes|   Shoes| Tiva|Green|  36|        70.86|     31.66|Female|
|         7|    Soft Linen Dress| Dresses| Tiva|White|   M|        57.66|     24.07|Female|
|         8| Soft Linen Trousers|   Pants| Tiva|Green|   M|        50.91|     33

In [247]:
joined_repartitioned_fashion_store_sales_and_salesitems_df=repartitioned_fashion_store_salesitems_df.join(repartitioned_fashion_store_sales_df,"sale_id")

In [248]:
joined_repartitioned_fashion_store_sales_and_salesitems_df.rdd.getNumPartitions()

1

**3.Bucket Join**

In [249]:
fashion_store_sales_df.rdd.getNumPartitions()

1

In [250]:
repartitoned_sales_df=fashion_store_sales_df.repartition(10,"sale_id")
repartitoned_salesitems_df=fashion_store_salesitems_df.repartition(10,"product_id")

In [252]:
repartitoned_sales_df.rdd.getNumPartitions()

10

In [253]:
repartitoned_salesitems_df.rdd.getNumPartitions()

10

**4.skew join**

In [255]:
skew_join=fashion_store_sales_df.join(fashion_store_salesitems_df.hint("skew"),"sale_id")

In [256]:
skew_join.printSchema()

root
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- channel: string (nullable = true)
 |-- channel_campaigns: string (nullable = true)



In [257]:
skew_join.show()

+-------+----------+----------+------------+----------+-----------+-----------+-------+----------+--------+--------------+----------+----------------+----------------+----------+----------+----------+----------+-----------------+
|sale_id|   channel|discounted|total_amount| sale_date|customer_id|    country|item_id|product_id|quantity|original_price|unit_price|discount_applied|discount_percent|discounted|item_total| sale_date|   channel|channel_campaigns|
+-------+----------+----------+------------+----------+-----------+-----------+-------+----------+--------+--------------+----------+----------------+----------------+----------+----------+----------+----------+-----------------+
|    658|App Mobile|         0|      374.25|2025-06-16|        835|   Portugal|   2270|       403|       1|          81.8|      81.8|             0.0|           0.00%|         0|      81.8|2025-06-16|App Mobile|       App Mobile|
|    336|E-commerce|         0|      536.47|2025-06-17|        790|     France| 

### **Data write operation into file system**

In [258]:
final_df_unique.write.option("header","true").mode("overwrite").csv("/content/csv_files/fashion_store_joined_dataframe/")


**read parquet files**

In [259]:
read_fashion_store_df=spark.read.option("header","true").option("inferschema","true").parquet("/content/parquet_files/fashion_store_joined_dataframe/")

In [260]:
read_fashion_store_df.printSchema()
read_fashion_store_df.show()

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- sale_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- discount_type: string (nullable = true)
 |-- discount_value: string (nullable = true)
 |-- description: string (nullable = true)
 |-- discounted: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- original_price: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- discounted_1: integer (nullable = true)
 |-- item_total: double (nullable = true)
 |-- sale_date_1: date (nullable = true)
 |-- chan